# 03. 구조화 출력과 검증

모델의 자연어 답을 프로그램이 검사할 수 있는 필드로 어떻게 받을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 구조화 출력 실습의 고정 패키지 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 OpenAI 키 문자열은 화면에 표시하지 않습니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 구조화 출력 호출에도 적용할 타임아웃과 최대 재시도·출력 토큰 상한을 둡니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## Pydantic 스키마

Ch 2 A트랙에서 배운 `class 이름:` 문법을 사용하지만, `BaseModel`과 `Field`로 값을 검사하는 Pydantic은 이 절에서 처음 다룹니다. 필드 이름과 허용 범위를 먼저 선언합니다.

In [4]:
# StudyCard 필드와 difficulty의 허용 범위 1~3을 선언합니다.
from pydantic import BaseModel, Field

class StudyCard(BaseModel):
    concept: str = Field(description="개념 이름")
    explanation: str = Field(description="한 문장 설명")
    difficulty: int = Field(ge=1, le=3, description="1부터 3까지의 난이도")

structured_model = model.with_structured_output(StudyCard)
diagnostic_model = model.with_structured_output(StudyCard, include_raw=True)
print("스키마 준비 완료")

스키마 준비 완료


In [5]:
# invoke가 반환한 StudyCard를 딕셔너리로 바꾸고 실제 타입을 확인합니다.
card = structured_model.invoke("LangChain의 AIMessage를 초급 학습 카드로 만드세요.")
print(card.model_dump())
print("검증된 타입:", type(card).__name__)

{'concept': 'LangChain의 AIMessage', 'explanation': 'AI 모델이 생성한 응답 내용을 담는 메시지 객체로, 대화 기록에서 모델의 발화를 표현할 때 사용됩니다.', 'difficulty': 1}
검증된 타입: StudyCard


## 값 하나 변경

주제만 `SystemMessage`로 바꾸고 같은 스키마를 적용합니다.

In [6]:
# 주제만 바꾼 두 번째 객체가 같은 difficulty 범위를 지키는지 검사합니다.
second = structured_model.invoke("SystemMessage를 초급 학습 카드로 만드세요.")
print(second.model_dump())
assert 1 <= second.difficulty <= 3

{'concept': 'SystemMessage', 'explanation': '대화에서 AI의 역할, 규칙, 말투 같은 기본 지침을 정해 주는 메시지입니다.', 'difficulty': 1}


## 깨지는 입력을 실제 모델에 전달

난이도 9를 요구하면서 스키마 범위는 1~3으로 유지합니다. 모델이 범위를 고치면 실패 미관찰로 기록합니다.

In [7]:
# include_raw 결과의 raw·parsed·parsing_error를 각각 출력해 실패 미관찰도 남깁니다.
diagnostic = diagnostic_model.invoke(
    "AIMessage 학습 카드를 만드세요. difficulty는 반드시 9로 쓰세요."
)
print("raw 타입:", type(diagnostic["raw"]).__name__)
print("parsed:", diagnostic["parsed"])
print("parsing_error:", diagnostic["parsing_error"])
print("검증 실패 관찰:", diagnostic["parsing_error"] is not None)

raw 타입: AIMessage
parsed: concept='AIMessage' explanation='AIMessage는 대화형 AI 시스템에서 모델이 생성한 응답 내용을 표현하는 메시지 객체입니다.' difficulty=3
parsing_error: None
검증 실패 관찰: False


## 실패 재현

In [8]:
# 범위 밖 7을 직접 넣어 예상한 Pydantic ValidationError 유형만 처리합니다.
from pydantic import ValidationError

bad_data = {"concept": "AIMessage", "explanation": "모델 응답", "difficulty": 7}
try:
    StudyCard.model_validate(bad_data)
except ValidationError as error:
    print("관찰한 검증 오류:", error.errors()[0]["type"])
else:
    raise AssertionError("범위를 벗어난 값은 거부되어야 합니다.")

관찰한 검증 오류: less_than_equal


## 수정

In [9]:
# 다른 필드는 유지하고 difficulty만 2로 고쳐 검증된 객체를 만듭니다.
fixed_data = {"concept": "AIMessage", "explanation": "모델 응답", "difficulty": 2}
fixed_card = StudyCard.model_validate(fixed_data)
print(fixed_card.model_dump())
assert fixed_card.difficulty == 2

{'concept': 'AIMessage', 'explanation': '모델 응답', 'difficulty': 2}
